# Generate and Append Novel SQL Tasks (Direct to `tests/golden_set1.json`)

This notebook is designed to:
1. Profile current golden set coverage
2. Generate novel, schema-aware NL-to-SQL tasks across DBMS topics
3. Validate and de-duplicate candidates
4. Append accepted tasks directly to `tests/golden_set1.json` with atomic write
5. Rebuild split artifacts and run evaluator smoke tests

## 1) Load Project Paths and Back Up `tests/golden_set1.json`

Loads the repository paths, opens the current `golden_set1` file, and writes a timestamped backup before any mutation.

In [ ]:
import os
import json
import shutil
import tempfile
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict


def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd.parent.parent]
    for c in candidates:
        if (c / "tests").exists() and (c / "scripts").exists():
            return c
    raise FileNotFoundError("Could not resolve repository root from current working directory")


REPO_ROOT = resolve_repo_root()
TESTS_DIR = REPO_ROOT / "tests"
DATA_DIR = REPO_ROOT / "data"
BACKUP_DIR = TESTS_DIR / "backups"
BACKUP_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_GOLDEN = TESTS_DIR / "golden_set.json"
TARGET_GOLDEN = TESTS_DIR / "golden_set1.json"
if not TARGET_GOLDEN.exists():
    if not SOURCE_GOLDEN.exists():
        raise FileNotFoundError(f"Missing source golden file: {SOURCE_GOLDEN}")
    shutil.copy2(SOURCE_GOLDEN, TARGET_GOLDEN)

with TARGET_GOLDEN.open("r") as f:
    golden_cases = json.load(f)

backup_ts = datetime.now().strftime("%Y%m%d_%H%M%S")
backup_file = BACKUP_DIR / f"golden_set1_backup_{backup_ts}.json"
shutil.copy2(TARGET_GOLDEN, backup_file)

print("Repository root:", REPO_ROOT)
print("Target golden set:", TARGET_GOLDEN)
print("Backup created:", backup_file)
print("Current case count:", len(golden_cases))

## 2) Load SQLite Schema and Build DBMS Topic Taxonomy

Extracts schema metadata and defines a broad DBMS topic taxonomy to drive generation coverage.

In [ ]:
import sqlite3

DB_PATH = DATA_DIR / "company_expanded.db"
if not DB_PATH.exists():
    fallback_db = DATA_DIR / "company.db"
    if not fallback_db.exists():
        raise FileNotFoundError(f"Neither {DB_PATH} nor {fallback_db} exists")
    DB_PATH = fallback_db

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name;")
TABLES = [r[0] for r in cur.fetchall()]

schema_meta = {}
for table in TABLES:
    cur.execute(f"PRAGMA table_info({table})")
    cols = cur.fetchall()
    cur.execute(f"PRAGMA foreign_key_list({table})")
    fks = cur.fetchall()
    schema_meta[table] = {
        "columns": [{"name": c[1], "type": c[2], "notnull": c[3], "pk": c[5]} for c in cols],
        "foreign_keys": [{"to_table": fk[2], "from_col": fk[3], "to_col": fk[4]} for fk in fks],
    }

DBMS_TOPIC_TAXONOMY = {
    "single_table_filters": "basic where clauses, predicates, sorting",
    "joins_inner_left_cross_self": "inner/left/cross/self joins",
    "group_by_having": "aggregations and filtered aggregates",
    "nested_subqueries": "IN/NOT IN nested query patterns",
    "correlated_subqueries": "row-wise dependent subqueries",
    "cte_and_recursive_cte": "WITH and WITH RECURSIVE",
    "window_functions": "OVER, running totals, partition analytics",
    "set_operations": "UNION/INTERSECT/EXCEPT",
    "case_expressions": "conditional logic in SQL",
    "date_time_ops": "strftime, julianday, date arithmetic",
    "string_ops": "LIKE, UPPER, LOWER, LENGTH, SUBSTR",
    "ranking": "ROW_NUMBER, RANK, DENSE_RANK",
    "pagination": "LIMIT/OFFSET patterns",
    "dml_safe_patterns": "read-only preview queries before updates",
    "transaction_aware_prompts": "transaction planning prompts with SELECT-only SQL",
}

print("DB path:", DB_PATH)
print("Tables:", TABLES)
print("Topic taxonomy size:", len(DBMS_TOPIC_TAXONOMY))
for t, desc in DBMS_TOPIC_TAXONOMY.items():
    print(f"- {t}: {desc}")

## 3) Profile Current Golden Set Coverage and Gaps

Infers topic tags from SQL, computes per-topic/per-difficulty counts, and identifies underrepresented areas.

In [ ]:
import re


def infer_topic_tags(sql: str):
    s = " ".join(sql.lower().strip().split())
    tags = set()
    if " join " in f" {s} ":
        tags.add("joins_inner_left_cross_self")
    if " group by " in f" {s} ":
        tags.add("group_by_having")
    if " having " in f" {s} ":
        tags.add("group_by_having")
    if " with " in f" {s} ":
        tags.add("cte_and_recursive_cte")
    if " recursive " in f" {s} ":
        tags.add("cte_and_recursive_cte")
    if " over (" in s:
        tags.add("window_functions")
    if " union " in f" {s} " or " intersect " in f" {s} " or " except " in f" {s} ":
        tags.add("set_operations")
    if " case " in f" {s} ":
        tags.add("case_expressions")
    if " strftime(" in s or " julianday(" in s or " date(" in s:
        tags.add("date_time_ops")
    if " like " in f" {s} " or " upper(" in s or " lower(" in s or " substr(" in s or " length(" in s:
        tags.add("string_ops")
    if "row_number()" in s or "rank()" in s or "dense_rank()" in s:
        tags.add("ranking")
    if " limit " in f" {s} " or " offset " in f" {s} ":
        tags.add("pagination")
    if " exists (" in s or " in (select " in s:
        tags.add("nested_subqueries")
    if "select" in s and not tags:
        tags.add("single_table_filters")
    return sorted(tags)


with TARGET_GOLDEN.open("r") as f:
    current_cases = json.load(f)

by_topic = Counter()
by_difficulty = Counter()
by_topic_difficulty = Counter()

for c in current_cases:
    diff = c.get("difficulty", "medium")
    tags = c.get("topic_tags") or infer_topic_tags(c.get("sql", ""))
    by_difficulty[diff] += 1
    for t in tags:
        by_topic[t] += 1
        by_topic_difficulty[(t, diff)] += 1

print("Current total cases:", len(current_cases))
print("By difficulty:", dict(by_difficulty))
print("\nTop topic counts:")
for topic, cnt in by_topic.most_common():
    print(f"- {topic}: {cnt}")

print("\nPotential gaps (topic, easy/medium/hard):")
for topic in DBMS_TOPIC_TAXONOMY:
    e = by_topic_difficulty[(topic, "easy")]
    m = by_topic_difficulty[(topic, "medium")]
    h = by_topic_difficulty[(topic, "hard")]
    if min(e, m, h) < 12:
        print(f"- {topic}: easy={e}, medium={m}, hard={h}")

## 4) Define Query Generation Quotas by Topic and Difficulty

Sets minimum quotas and hard caps for each topic/difficulty pair to keep the final set balanced and realistic.

In [ ]:
TARGET_TOTAL = 1000
current_total = len(current_cases)
max_to_add = max(TARGET_TOTAL - current_total, 0)

topics = list(DBMS_TOPIC_TAXONOMY.keys())
difficulties = ["easy", "medium", "hard"]

min_base = max(6, max_to_add // (len(topics) * len(difficulties)))
quota_table = {}
hard_cap_table = {}
for t in topics:
    for d in difficulties:
        quota_table[(t, d)] = min_base
        hard_cap_table[(t, d)] = max(30, min_base * 4)

for t in ["joins_inner_left_cross_self", "group_by_having", "nested_subqueries", "cte_and_recursive_cte", "window_functions", "set_operations"]:
    quota_table[(t, "medium")] += 6
    quota_table[(t, "hard")] += 8

constraints = {
    "target_total": TARGET_TOTAL,
    "max_to_add": max_to_add,
    "strict_uniqueness": "No duplicate normalized question or SQL",
    "schema_realism": "All SQL must execute on DB and use existing schema",
    "distribution": "Balanced coverage across topic x difficulty",
}

print("Current total:", current_total)
print("Target total:", TARGET_TOTAL)
print("Max add:", max_to_add)
print("Sample quotas:")
for k in list(quota_table.keys())[:10]:
    print(k, "min", quota_table[k], "cap", hard_cap_table[k])
print("Constraints:", constraints)

## 5) Generate Novel NL-to-SQL Candidates

Uses deterministic template generation (and leaves optional model-assisted hook) to produce schema-aware candidates.

In [ ]:
import importlib.util
import itertools

build_script = REPO_ROOT / "scripts" / "build_golden_set1.py"
if not build_script.exists():
    raise FileNotFoundError(f"Missing generation script: {build_script}")

spec = importlib.util.spec_from_file_location("golden_builder", build_script)
golden_builder = importlib.util.module_from_spec(spec)
if spec is None or spec.loader is None:
    raise RuntimeError("Failed to load build_golden_set1 module")
spec.loader.exec_module(golden_builder)

conn_gen = sqlite3.connect(DB_PATH)
raw_candidates = list(itertools.islice(golden_builder.candidate_pool(conn_gen.cursor()), 200000))
conn_gen.close()

print("Raw candidate count:", len(raw_candidates))
print("Candidate schema keys:", sorted(raw_candidates[0].keys()) if raw_candidates else "(none)")

# Optional model-assisted generation hook (disabled by default)
ENABLE_MODEL_ASSISTED = False
if ENABLE_MODEL_ASSISTED:
    print("Model-assisted generation is enabled; add your LLM generation pipeline here.")
else:
    print("Model-assisted generation disabled; using deterministic templates only.")

## 6) Enforce Uniqueness and Remove Near-Duplicates

Normalizes SQL and question text, removes exact duplicates, and filters near-duplicate phrasing.

In [ ]:
def normalize_question(q: str) -> str:
    return golden_builder.norm_text(q)


def normalize_sql(sql: str) -> str:
    return golden_builder.canonical_sql(sql)


existing_q = {normalize_question(c["question"]) for c in current_cases}
existing_sql = {normalize_sql(c["sql"]) for c in current_cases}
recent_q_tokens = [golden_builder.token_set(c["question"]) for c in current_cases]

unique_candidates = []
seen_local_q = set()
seen_local_sql = set()

for c in raw_candidates:
    q = c["question"].strip()
    s = c["sql"].strip()
    qn = normalize_question(q)
    sn = normalize_sql(s)

    if qn in existing_q or qn in seen_local_q:
        continue
    if sn in existing_sql or sn in seen_local_sql:
        continue

    q_tok = golden_builder.token_set(q)
    if any(golden_builder.jaccard(q_tok, prev) > 0.985 for prev in recent_q_tokens[-300:]):
        continue

    seen_local_q.add(qn)
    seen_local_sql.add(sn)
    recent_q_tokens.append(q_tok)
    unique_candidates.append(c)

print("Candidates after uniqueness filters:", len(unique_candidates))

## 7) Validate SQL Syntax, Execution, and Result Safety

Executes candidates in a sandboxed SQLite connection with timeout checks, rejects unsafe SQL, and filters unstable outputs.

In [ ]:
quota_counts = Counter()
accepted_candidates = []
rejected_candidates = []

conn_val = sqlite3.connect(DB_PATH)

for cand in unique_candidates:
    if len(accepted_candidates) >= max_to_add:
        break

    question = cand["question"].strip()
    sql = cand["sql"].strip()
    difficulty = cand.get("difficulty", "medium")
    tags = cand.get("topic_tags") or golden_builder.infer_topic_tags(sql)

    if not golden_builder.is_safe_sql(sql):
        rejected_candidates.append({"question": question, "reason": "unsafe_sql"})
        continue

    # Enforce per-topic hard cap to avoid over-concentration.
    cap_breach = False
    for t in tags:
        if quota_counts[(t, difficulty)] >= hard_cap_table.get((t, difficulty), 999999):
            cap_breach = True
            break
    if cap_breach:
        rejected_candidates.append({"question": question, "reason": "topic_hard_cap"})
        continue

    try:
        rows1 = golden_builder.execute_with_timeout(conn_val, sql, timeout_ms=1500)
        rows2 = golden_builder.execute_with_timeout(conn_val, sql, timeout_ms=1500)
    except Exception as e:
        rejected_candidates.append({"question": question, "reason": f"exec_error:{type(e).__name__}"})
        continue

    if golden_builder.normalise_rows(rows1) != golden_builder.normalise_rows(rows2):
        rejected_candidates.append({"question": question, "reason": "nondeterministic_result"})
        continue

    if rows1 is None:
        rejected_candidates.append({"question": question, "reason": "invalid_result"})
        continue

    accepted_candidates.append(cand)
    for t in tags:
        quota_counts[(t, difficulty)] += 1

conn_val.close()

print("Accepted validated candidates:", len(accepted_candidates))
print("Rejected candidates:", len(rejected_candidates))

## 8) Auto-Label Queries (joins, subqueries, CTE, window, aggregation, set ops, DML)

Adds rule-based tags for each accepted query for downstream coverage analysis and stratified splitting.

In [ ]:
for cand in accepted_candidates:
    tags = cand.get("topic_tags") or golden_builder.infer_topic_tags(cand["sql"])
    cand["topic_tags"] = sorted(set(tags))
    if not cand.get("category"):
        cand["category"] = "generated"

print("Sample tagged candidates:")
for s in accepted_candidates[:5]:
    print("-", s["difficulty"], s["category"], s["topic_tags"], "|", s["question"][:80])

## 9) Append Accepted Cases Directly to `tests/golden_set1.json`

Appends validated candidates to the in-memory corpus and writes atomically to `tests/golden_set1.json` (non-destructive to original).

In [ ]:
with TARGET_GOLDEN.open("r") as f:
    target_cases = json.load(f)

start_id = max((c.get("id", 0) for c in target_cases), default=0) + 1
room = max(TARGET_TOTAL - len(target_cases), 0)
to_append = accepted_candidates[:room]

newly_appended_cases = []
for i, c in enumerate(to_append):
    item = {
        "id": start_id + i,
        "difficulty": c.get("difficulty", "medium"),
        "category": c.get("category", "generated"),
        "tables_used": c.get("tables_used", []),
        "topic_tags": c.get("topic_tags") or infer_topic_tags(c.get("sql", "")),
        "question": c["question"].strip(),
        "sql": c["sql"].strip(),
    }
    target_cases.append(item)
    newly_appended_cases.append(item)

target_cases = sorted(target_cases, key=lambda x: x["id"])

tmp_file = TARGET_GOLDEN.with_suffix(".json.tmp")
with tmp_file.open("w") as f:
    json.dump(target_cases, f, indent=2)
tmp_file.replace(TARGET_GOLDEN)

print("Appended:", len(newly_appended_cases))
print("New total:", len(target_cases))
print("Wrote file:", TARGET_GOLDEN)

## 10) Rebuild Split Files and Coverage Reports

Regenerates train/val/test splits with stratification over `(difficulty, category)` and prints before/after coverage summaries.

In [ ]:
import random

with TARGET_GOLDEN.open("r") as f:
    all_cases_after = json.load(f)

random.seed(42)
groups = defaultdict(list)
for item in all_cases_after:
    key = (item.get("difficulty", "medium"), item.get("category", "generated"))
    groups[key].append(item)

train, val, test = [], [], []
for _, items in groups.items():
    random.shuffle(items)
    n = len(items)
    n_train = max(1, int(0.70 * n)) if n > 2 else max(1, n - 1)
    n_val = int(0.15 * n) if n > 6 else (1 if n > 3 else 0)
    if n_train + n_val >= n:
        n_val = max(0, n - n_train - 1)
    n_test = n - n_train - n_val

    train.extend(items[:n_train])
    val.extend(items[n_train:n_train + n_val])
    test.extend(items[n_train + n_val:n_train + n_val + n_test])

random.shuffle(train)
random.shuffle(val)
random.shuffle(test)

train_path = TESTS_DIR / "golden_set1_train.json"
val_path = TESTS_DIR / "golden_set1_val.json"
test_path = TESTS_DIR / "golden_set1_test.json"

for path, payload in [(train_path, train), (val_path, val), (test_path, test)]:
    with path.open("w") as f:
        json.dump(payload, f, indent=2)

print("Split sizes:", {"train": len(train), "val": len(val), "test": len(test)})
print("Saved:", train_path, val_path, test_path)

# Coverage before/after snapshot.
before_topics = Counter()
after_topics = Counter()
for c in current_cases:
    for t in (c.get("topic_tags") or infer_topic_tags(c.get("sql", ""))):
        before_topics[t] += 1
for c in all_cases_after:
    for t in (c.get("topic_tags") or infer_topic_tags(c.get("sql", ""))):
        after_topics[t] += 1

print("\nCoverage delta by topic:")
all_topic_keys = sorted(set(before_topics.keys()) | set(after_topics.keys()))
for t in all_topic_keys:
    b = before_topics[t]
    a = after_topics[t]
    print(f"- {t}: before={b}, after={a}, delta={a-b}")

## 11) Run Evaluator Smoke Test on Newly Appended Cases

Runs evaluator on a sampled subset of new cases (if runtime is configured) and exports QA artifacts with acceptance/rejection reasons.

In [ ]:
import csv

results_dir = TESTS_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)
qa_csv = results_dir / f"golden_set1_qa_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"

with qa_csv.open("w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["bucket", "question", "reason"])
    writer.writeheader()
    for c in newly_appended_cases:
        writer.writerow({"bucket": "accepted", "question": c["question"], "reason": ""})
    for r in rejected_candidates[:2000]:
        writer.writerow({"bucket": "rejected", "question": r.get("question", ""), "reason": r.get("reason", "")})

print("QA CSV exported:", qa_csv)

# Evaluator smoke test (best effort): this requires your LLM runtime/config to be available.
smoke_sample = newly_appended_cases[: min(20, len(newly_appended_cases))]
print("Smoke sample size:", len(smoke_sample))

if smoke_sample:
    try:
        import tests.evaluator as evaluator

        evaluator.DB_PATH = str(DB_PATH)
        evaluator.GOLDEN_SET = str(TARGET_GOLDEN)
        evaluator.SLEEP_BETWEEN = 0.0

        smoke_report = evaluator.run_evaluation(smoke_sample)
        print("Smoke execution accuracy (relaxed):", smoke_report.get("execution_accuracy"))
        print("Smoke execution accuracy (strict):", smoke_report.get("strict_execution_accuracy"))
        print("Smoke partial-credit accuracy:", smoke_report.get("partial_credit_accuracy"))
    except Exception as e:
        print("Evaluator smoke test skipped/fail (environment-dependent):", repr(e))
        print("You can still use QA CSV + SQL validation logs from previous cells.")
else:
    print("No new cases were appended in this run, so smoke evaluation was skipped.")